## 1. Database Setup

First, we'll import the `sqlite3` library and connect to our database. If the database file doesn't exist, it will be created.

In [ ]:
import sqlite3

# Connect to SQLite database (creates a new database if it doesn't exist)
conn = sqlite3.connect('Employee_Management_DB.db')
cursor = conn.cursor()

print("Database 'Employee_Management_DB.db' created and connected successfully.")

## 2. Table Creation

Now, we'll create the tables for our Employee Management System with appropriate data types and constraints.

In [ ]:
# Create Departments Table
cursor.execute('''
CREATE TABLE IF NOT EXISTS Departments (
    department_id INTEGER PRIMARY KEY AUTOINCREMENT,
    department_name TEXT NOT NULL UNIQUE
);
''')

# Create Jobs Table
cursor.execute('''
CREATE TABLE IF NOT EXISTS Jobs (
    job_id INTEGER PRIMARY KEY AUTOINCREMENT,
    job_title TEXT NOT NULL UNIQUE,
    min_salary DECIMAL(10, 2) CHECK (min_salary >= 0),
    max_salary DECIMAL(10, 2) CHECK (max_salary >= min_salary)
);
''')

# Create Employees Table
cursor.execute('''
CREATE TABLE IF NOT EXISTS Employees (
    employee_id INTEGER PRIMARY KEY AUTOINCREMENT,
    first_name TEXT NOT NULL,
    last_name TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    phone_number TEXT,
    hire_date DATE NOT NULL DEFAULT CURRENT_DATE,
    job_id INTEGER,
    salary DECIMAL(10, 2) CHECK (salary >= 0),
    department_id INTEGER,
    manager_id INTEGER,
    FOREIGN KEY (job_id) REFERENCES Jobs(job_id),
    FOREIGN KEY (department_id) REFERENCES Departments(department_id),
    FOREIGN KEY (manager_id) REFERENCES Employees(employee_id)
);
''')

# Create Salaries Table (to track salary history if needed, though 'salary' in Employees is for current)
cursor.execute('''
CREATE TABLE IF NOT EXISTS Salaries (
    salary_id INTEGER PRIMARY KEY AUTOINCREMENT,
    employee_id INTEGER NOT NULL,
    amount DECIMAL(10, 2) NOT NULL CHECK (amount >= 0),
    from_date DATE NOT NULL,
    to_date DATE,
    FOREIGN KEY (employee_id) REFERENCES Employees(employee_id)
);
''')

# Create Attendance Table
cursor.execute('''
CREATE TABLE IF NOT EXISTS Attendance (
    attendance_id INTEGER PRIMARY KEY AUTOINCREMENT,
    employee_id INTEGER NOT NULL,
    attendance_date DATE NOT NULL,
    status TEXT NOT NULL CHECK (status IN ('Present', 'Absent', 'Late', 'Leave')),
    FOREIGN KEY (employee_id) REFERENCES Employees(employee_id),
    UNIQUE (employee_id, attendance_date)
);
''')

# Create Projects Table
cursor.execute('''
CREATE TABLE IF NOT EXISTS Projects (
    project_id INTEGER PRIMARY KEY AUTOINCREMENT,
    project_name TEXT NOT NULL UNIQUE,
    start_date DATE NOT NULL,
    end_date DATE CHECK (end_date >= start_date),
    status TEXT DEFAULT 'Pending' CHECK (status IN ('Pending', 'In Progress', 'Completed', 'Cancelled'))
);
''')

# Create Employee_Projects Table (Many-to-Many relationship)
cursor.execute('''
CREATE TABLE IF NOT EXISTS Employee_Projects (
    employee_id INTEGER NOT NULL,
    project_id INTEGER NOT NULL,
    PRIMARY KEY (employee_id, project_id),
    FOREIGN KEY (employee_id) REFERENCES Employees(employee_id),
    FOREIGN KEY (project_id) REFERENCES Projects(project_id)
);
''')

conn.commit()
print("All tables created successfully.")

## 3. Data Insertion

Let's populate our tables with some sample data.

In [ ]:
# Insert into Departments
departments_data = [
    ('Human Resources',),
    ('Engineering',),
    ('Sales',),
    ('Marketing',),
    ('Finance',)
]
cursor.executemany("INSERT INTO Departments (department_name) VALUES (?)", departments_data)

# Insert into Jobs
jobs_data = [
    ('HR Manager', 60000, 90000),
    ('Software Engineer', 70000, 120000),
    ('Sales Representative', 40000, 75000),
    ('Marketing Specialist', 50000, 80000),
    ('Financial Analyst', 55000, 95000),
    ('Project Manager', 80000, 130000),
    ('CEO', 150000, 300000)
]
cursor.executemany("INSERT INTO Jobs (job_title, min_salary, max_salary) VALUES (?, ?, ?)", jobs_data)

# Insert into Employees (using current_date for hire_date, and job_id, department_id references)
# Note: manager_id will be added in a separate update after employees exist
employees_data = [
    ('Alice', 'Smith', 'alice.s@example.com', '111-222-3333', '2022-01-15', 7, 200000, 1, None), # CEO
    ('Bob', 'Johnson', 'bob.j@example.com', '111-222-4444', '2022-03-01', 1, 75000, 1, 1), # HR Manager
    ('Charlie', 'Brown', 'charlie.b@example.com', '111-222-5555', '2021-06-20', 2, 90000, 2, 1), # Software Engineer
    ('Diana', 'Prince', 'diana.p@example.com', '111-222-6666', '2022-02-10', 3, 60000, 3, 1), # Sales Rep
    ('Eve', 'Adams', 'eve.a@example.com', '111-222-7777', '2023-04-01', 4, 65000, 4, 1), # Marketing Spec
    ('Frank', 'White', 'frank.w@example.com', '111-222-8888', '2022-09-01', 5, 80000, 5, 1), # Financial Analyst
    ('Grace', 'Lee', 'grace.l@example.com', '111-222-9999', '2023-01-01', 2, 85000, 2, 3) # Software Engineer
]
cursor.executemany("INSERT INTO Employees (first_name, last_name, email, phone_number, hire_date, job_id, salary, department_id, manager_id) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)", employees_data)

# Update manager_id for existing employees (e.g., set manager for Bob, Charlie, etc.)
# For simplicity, Alice (employee_id 1) is the manager for employees 2-6.
# Grace (employee_id 7) is managed by Charlie (employee_id 3).
cursor.execute("UPDATE Employees SET manager_id = 1 WHERE employee_id IN (2, 3, 4, 5, 6)")
cursor.execute("UPDATE Employees SET manager_id = 3 WHERE employee_id = 7")

# Insert into Salaries (example: current salaries)
salaries_data = [
    (1, 200000, '2022-01-15', None),
    (2, 75000, '2022-03-01', None),
    (3, 90000, '2021-06-20', None),
    (4, 60000, '2022-02-10', None),
    (5, 65000, '2023-04-01', None),
    (6, 80000, '2022-09-01', None),
    (7, 85000, '2023-01-01', None)
]
cursor.executemany("INSERT INTO Salaries (employee_id, amount, from_date, to_date) VALUES (?, ?, ?, ?)", salaries_data)

# Insert into Attendance
attendance_data = [
    (1, '2023-10-26', 'Present'), (1, '2023-10-25', 'Present'),
    (2, '2023-10-26', 'Present'), (2, '2023-10-25', 'Late'),
    (3, '2023-10-26', 'Present'), (3, '2023-10-25', 'Leave'),
    (4, '2023-10-26', 'Present'), (4, '2023-10-25', 'Present'),
    (5, '2023-10-26', 'Absent'), (5, '2023-10-25', 'Present')
]
cursor.executemany("INSERT INTO Attendance (employee_id, attendance_date, status) VALUES (?, ?, ?)", attendance_data)

# Insert into Projects
projects_data = [
    ('HR System Upgrade', '2023-01-01', '2023-12-31', 'In Progress'),
    ('New Product Launch', '2023-05-15', '2024-03-31', 'In Progress'),
    ('Sales Q4 Campaign', '2023-09-01', '2023-11-30', 'Completed'),
    ('Financial Audit 2023', '2023-08-01', '2023-10-31', 'Completed')
]
cursor.executemany("INSERT INTO Projects (project_name, start_date, end_date, status) VALUES (?, ?, ?, ?)", projects_data)

# Insert into Employee_Projects
employee_projects_data = [
    (1, 1), (1, 2), # Alice is involved in HR System and New Product Launch
    (2, 1),         # Bob is involved in HR System
    (3, 2), (3, 1), # Charlie is involved in New Product Launch and HR System
    (7, 2),         # Grace is involved in New Product Launch
    (4, 3),         # Diana is involved in Sales Q4 Campaign
    (5, 2),         # Eve is involved in New Product Launch
    (6, 4)          # Frank is involved in Financial Audit
]
cursor.executemany("INSERT INTO Employee_Projects (employee_id, project_id) VALUES (?, ?)", employee_projects_data)

conn.commit()
print("Sample data inserted successfully.")

## 4. Basic SQL Queries

Let's run some basic `SELECT` queries to verify our data.

In [ ]:
print("\n--- All Employees ---")
cursor.execute("SELECT employee_id, first_name, last_name, email, salary FROM Employees;")
for row in cursor.fetchall():
    print(row)

print("\n--- All Departments ---")
cursor.execute("SELECT * FROM Departments;")
for row in cursor.fetchall():
    print(row)

print("\n--- Employees in Engineering Department ---")
cursor.execute("""
SELECT E.first_name, E.last_name, D.department_name
FROM Employees E
JOIN Departments D ON E.department_id = D.department_id
WHERE D.department_name = 'Engineering';
""")
for row in cursor.fetchall():
    print(row)

print("\n--- Employees and their Jobs ---")
cursor.execute("""
SELECT E.first_name, E.last_name, J.job_title
FROM Employees E
JOIN Jobs J ON E.job_id = J.job_id;
""")
for row in cursor.fetchall():
    print(row)

## 5. Close Connection

Remember to close the database connection when you are done.

In [ ]:
conn.close()
print("Database connection closed.")

## 6. Project Deliverables and Documentation

### ER Diagram (Conceptual Description)

An Entity-Relationship (ER) Diagram visually represents the structure of the database and the relationships between its tables. For this `Employee_Management_DB`, a conceptual ER diagram would show:

*   **Entities (Tables):** Departments, Employees, Jobs, Salaries, Attendance, Projects, Employee_Projects.
*   **Attributes (Columns):** Each entity box would list its respective columns, with primary keys (PK) underlined.
*   **Relationships:**
    *   `Employees` (Many) to `Departments` (One): An employee belongs to one department. (FK: `department_id` in Employees referencing `department_id` in Departments)
    *   `Employees` (Many) to `Jobs` (One): An employee holds one job. (FK: `job_id` in Employees referencing `job_id` in Jobs)
    *   `Employees` (Many) to `Employees` (One): An employee can have one manager (who is also an employee). (FK: `manager_id` in Employees referencing `employee_id` in Employees)
    *   `Salaries` (Many) to `Employees` (One): Salary records belong to one employee. (FK: `employee_id` in Salaries referencing `employee_id` in Employees)
    *   `Attendance` (Many) to `Employees` (One): Attendance records belong to one employee. (FK: `employee_id` in Attendance referencing `employee_id` in Employees)
    *   `Employee_Projects` (Many-to-Many junction table) between `Employees` and `Projects`: An employee can work on multiple projects, and a project can have multiple employees. (FKs: `employee_id` in Employee_Projects referencing `employee_id` in Employees; `project_id` in Employee_Projects referencing `project_id` in Projects)

Tools like draw.io, dbdiagram.io, or even specific database management tools can be used to generate a visual ER diagram from this schema.

### Schema Summary

Here's a summary of the database schema:

*   **Departments:** Manages company departments.
    *   `department_id` (PK, INTEGER, AUTOINCREMENT)
    *   `department_name` (TEXT, NOT NULL, UNIQUE)

*   **Jobs:** Defines job roles and salary ranges.
    *   `job_id` (PK, INTEGER, AUTOINCREMENT)
    *   `job_title` (TEXT, NOT NULL, UNIQUE)
    *   `min_salary` (DECIMAL, CHECK >= 0)
    *   `max_salary` (DECIMAL, CHECK >= `min_salary`)

*   **Employees:** Stores employee personal and professional details.
    *   `employee_id` (PK, INTEGER, AUTOINCREMENT)
    *   `first_name` (TEXT, NOT NULL)
    *   `last_name` (TEXT, NOT NULL)
    *   `email` (TEXT, NOT NULL, UNIQUE)
    *   `phone_number` (TEXT)
    *   `hire_date` (DATE, NOT NULL, DEFAULT CURRENT_DATE)
    *   `job_id` (FK to Jobs, INTEGER)
    *   `salary` (DECIMAL, CHECK >= 0)
    *   `department_id` (FK to Departments, INTEGER)
    *   `manager_id` (FK to Employees, INTEGER, self-referencing)

*   **Salaries:** Tracks historical salary information for employees.
    *   `salary_id` (PK, INTEGER, AUTOINCREMENT)
    *   `employee_id` (FK to Employees, INTEGER, NOT NULL)
    *   `amount` (DECIMAL, NOT NULL, CHECK >= 0)
    *   `from_date` (DATE, NOT NULL)
    *   `to_date` (DATE)

*   **Attendance:** Records daily attendance status of employees.
    *   `attendance_id` (PK, INTEGER, AUTOINCREMENT)
    *   `employee_id` (FK to Employees, INTEGER, NOT NULL)
    *   `attendance_date` (DATE, NOT NULL)
    *   `status` (TEXT, NOT NULL, CHECK IN ('Present', 'Absent', 'Late', 'Leave'))
    *   `UNIQUE (employee_id, attendance_date)`

*   **Projects:** Manages project details.
    *   `project_id` (PK, INTEGER, AUTOINCREMENT)
    *   `project_name` (TEXT, NOT NULL, UNIQUE)
    *   `start_date` (DATE, NOT NULL)
    *   `end_date` (DATE, CHECK >= `start_date`)
    *   `status` (TEXT, DEFAULT 'Pending', CHECK IN ('Pending', 'In Progress', 'Completed', 'Cancelled'))

*   **Employee_Projects:** Links employees to projects (many-to-many relationship).
    *   `employee_id` (PK, FK to Employees, INTEGER, NOT NULL)
    *   `project_id` (PK, FK to Projects, INTEGER, NOT NULL)

### Relationship List

The database has the following key relationships:

*   **One-to-Many:**
    *   `Departments` (One) to `Employees` (Many): Each department can have multiple employees.
    *   `Jobs` (One) to `Employees` (Many): Each job title can be held by multiple employees.
    *   `Employees` (One) to `Salaries` (Many): Each employee can have multiple salary records (history).
    *   `Employees` (One) to `Attendance` (Many): Each employee can have multiple attendance records.
*   **Many-to-One (Self-Referencing):**
    *   `Employees` (Many) to `Employees` (One - Manager): An employee can be managed by another employee.
*   **Many-to-Many (via Junction Table):**
    *   `Employees` (Many) to `Projects` (Many): Implemented through the `Employee_Projects` junction table, allowing employees to work on multiple projects and projects to involve multiple employees.

### SQL Concepts Summary

This project demonstrates various SQL concepts:

*   **Data Definition Language (DDL):**
    *   `CREATE TABLE`: To define new tables.
    *   `PRIMARY KEY`: Uniquely identifies each record in a table.
    *   `FOREIGN KEY`: Enforces referential integrity, linking tables together.
    *   `NOT NULL`: Ensures that a column cannot have a NULL value.
    *   `UNIQUE`: Ensures all values in a column are different.
    *   `CHECK`: Enforces domain integrity by limiting the values that can be placed in a column.
    *   `DEFAULT`: Provides a default value for a column when none is specified.
    *   `AUTOINCREMENT`: Automatically generates a unique number for new records (often used with PRIMARY KEY).
*   **Data Manipulation Language (DML):**
    *   `INSERT INTO`: To add new records to a table.
    *   `UPDATE`: To modify existing records in a table.
    *   `SELECT`: To retrieve data from one or more tables.
    *   `JOIN` (INNER JOIN): To combine rows from two or more tables based on a related column between them.
    *   `WHERE`: To filter records based on specified conditions.
*   **Other Concepts:**
    *   `sqlite3` Python module for interacting with SQLite databases.
    *   `conn.commit()`: To save changes made to the database.
    *   `cursor.execute()` and `cursor.executemany()`: To run SQL commands.

### Future Enhancements

This project can be expanded with several enhancements:

1.  **More Advanced Queries:** Implement complex queries involving aggregations (`GROUP BY`, `HAVING`), subqueries, views, and stored procedures (though less common in SQLite).
2.  **Indexing:** Add indexes to frequently queried columns (e.g., `last_name`, `hire_date`, foreign keys) to improve query performance.
3.  **Triggers:** Create triggers for automated actions, such as updating a `last_update_date` column on record modification or enforcing complex business rules.
4.  **Reporting:** Integrate with data visualization libraries (e.g., Matplotlib, Seaborn, Plotly) to generate reports on employee demographics, salary distribution, project progress, and attendance trends.
5.  **User Interface:** Develop a simple web-based UI (e.g., using Flask or Django) or a desktop application to interact with the database more intuitively.
6.  **Error Handling:** Implement robust error handling for database operations.
7.  **Data Validation:** Add more sophisticated data validation logic in the application layer before inserting data into the database.
8.  **Historical Data Tracking:** Expand the `Salaries` table to be more comprehensive, and potentially add similar historical tables for `Jobs` or `Departments` if those change frequently.
9.  **Security:** For a production system, consider user roles and permissions, although SQLite's built-in security is limited for multi-user environments.
10. **Backup and Restore:** Implement scripts for regular database backups and recovery procedures.